# NB15 — equivalence tests for the null claims; PSWE threshold/duration grid (A11) and collinearity (CPU)

TOST (+/-0.05 macro-F1, Nadeau-Bengio corrected) and minimum detectable effects for every 'no difference' comparison, from the committed runs; the planned A11 PSWE rule grid (5/6/7 Hz x 3/5/8 s) and a collinearity analysis of the slowing adjustment, in both cohorts.

In [ ]:
REPO = 'https://github.com/Devguru-codes/Major_project_IIITN.git'
BRANCH = 'feat/implementation'
WORK = '/kaggle/working'
SRC = '/tmp/eegrep-src'   # code checkout stays out of the saved output

import subprocess, sys, os, json, platform, shutil

def sh(cmd, log=None):
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-20000:], r.stderr[-5000:], sep='\n', flush=True)
    if log:
        open(log, 'w').write(r.stdout + '\n' + r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f'command failed ({r.returncode}): {cmd}')
    return r.stdout

shutil.rmtree(SRC, ignore_errors=True)
sh(f'git clone --depth 1 -b {BRANCH} {REPO} {SRC}')
SHA = sh(f'git -C {SRC} rev-parse --short HEAD').strip()
sh(f'pip install -q -e "{SRC}[dev,stats]"', log=f'{WORK}/pip_install.log')
PY = f'cd {SRC} && {sys.executable} -m eegrep'
print('git sha', SHA)

In [ ]:
sh(f'cd {SRC} && {sys.executable} -m pytest -q -p no:cacheprovider tests/test_equivalence.py tests/test_pswe.py tests/test_stats.py', log=f'{WORK}/pytest_nb15.txt')

In [ ]:
sh(f'{PY} equivalence --reports {SRC}/reports --out {WORK}/equivalence', log=f'{WORK}/equivalence.log')

In [ ]:
import glob
pre = [d for d in glob.glob('/kaggle/input/**/preproc', recursive=True) if glob.glob(f'{d}/sub-*.npz')]
caches = [d for d in glob.glob('/kaggle/input/**/cache', recursive=True) if os.path.exists(f'{d}/index.npz')]
parts = glob.glob('/kaggle/input/**/participants.tsv', recursive=True)
PRE04 = sorted(d for d in pre if 'nb01' in d); PRE84 = [d for d in pre if 'nb11a' in d]
CACHE04 = [d for d in caches if 'nb02' in d][0]; CACHE84 = [d for d in caches if 'nb11a' in d][0]
PART04 = [p for p in parts if 'nb02' in p][0]; PART84 = [p for p in parts if 'nb11a' in p][0]
assert len(PRE04) == 3 and len(PRE84) == 1

In [ ]:
sh(f'{PY} pswe-grid --preproc {" ".join(PRE04)} --cache {CACHE04} --participants {PART04} --out {WORK}/pswe_grid_ds004504', log=f'{WORK}/pswe_grid_ds004504.log')

In [ ]:
sh(f'{PY} --config configs/ds004584.yaml pswe-grid --preproc {PRE84[0]} --cache {CACHE84} --participants {PART84} --out {WORK}/pswe_grid_ds004584', log=f'{WORK}/pswe_grid_ds004584.log')

In [ ]:
import torch
sh(f'{sys.executable} -m pip freeze', log=f'{WORK}/pip_freeze.txt')
env = {'git_sha': SHA, 'python': platform.python_version(), 'torch': torch.__version__,
       'cuda': torch.cuda.is_available(), 'cpu_count': os.cpu_count(),
       'kaggle_image': os.environ.get('KAGGLE_DOCKER_IMAGE', 'unknown')}
json.dump(env, open(f'{WORK}/environment.json', 'w'), indent=1)
print(env)